# Temperature data EDA (for specific date and time range)

In [ ]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.dates import DateFormatter, HourLocator

from temp_data_plot_functions import plot_hot_water, plot_room_temp, plot_tariff

In [ ]:
# Read in, sort and store data to use for problem
temp_path = Path("data/PSTData5")
temp_files = temp_path.glob("*.csv")

houses_temps = {}
zone_2 = True
for file in temp_files:
    if "Events" not in str(file):
        house_id = file.stem.split("_")[0]
        data = pd.read_csv(file)
        # Store data
        data["Datetime"] = pd.to_datetime(data["Time (UTC)"])
        houses_temps[house_id] = data


In [ ]:
house_id = "30055" #"30436" with start_index=22466
temp_df = houses_temps[house_id]

In [ ]:
# Slice paramters
start_index = 1871  # note: easiest to extract this as the row index in the excel file... could be automated
num_hours = 30  # Number of hours to plot
end_index = int(start_index+(num_hours*60/5))

In [ ]:
def set_up_figure(df, house_id, start_index, end_index):
    """Set up figure with subplots for plotting
    
    Args:
        df (Dataframe): dataframe of a house to plot
        house_id (str): id of house to plot
        start_index (int): index at start of data to begin plot from
        end_index (int): index at end of data to end plot at

    Returns:
        temp_plot (matplotlib subplot): subplot to plot room temperature variables
        hot_water_plot (matplotlib subplot): subplot to plot hot water variables
        tariff_plot (matplotlib subplot): subplot to plot tariff
    
    """
    fig, (temp_plot, hot_water_plot, tariff_plot) = plt.subplots(3, 1, sharex=True, height_ratios=[5, 5, 3], figsize=(15,6), layout="tight")
    plt.xlim(df["Datetime"][start_index],df["Datetime"][end_index])
    fig.suptitle(f"House: {house_id}")
    plt.xlabel("Date")
    ax = fig.gca()
    ax.xaxis.set_major_formatter(DateFormatter("%d %b %H:%M"))
    ax.xaxis.set_major_locator(HourLocator(interval=3))
    return temp_plot, hot_water_plot, tariff_plot

In [ ]:
def plot_temp_data(df, house_id, start_index, end_index):
    """Plot the temperature data by either week or day

    Args:
        df (Dataframe): data for plotting
        house_id (str): id of house to plot
        start_index (int): index at start of data to begin plot from
        end_index (int): index at end of data to end plot at

    Returns:
        Plots of the temperature data for a specific set of hours
    """
    # Get data
    dates = df["Datetime"][start_index:end_index]
    room_temp = df["Room temperature (Zone 1) (°C)"][start_index:end_index]
    user_setpoint = df["Setpoint temperature (Zone 1) (°C)"][start_index:end_index]
    flow_temp = df["Flow temperature (°C)"][start_index:end_index]
    hot_water_temp = df["Hot water temperature (°C)"][start_index:end_index]
    hot_water_setpoint = df["Hot water setpoint (°C)"][start_index:end_index]
    try: 
        ext_temp = df["External temperature (°C)"][start_index:end_index]
    except:  # noqa: E722
        ext_temp = None
    tariff = df["Tariff rate (p/kWh)"][start_index:end_index]

    # Make subfigs
    temp_plot, hot_water_plot, tariff_plot = set_up_figure(df, house_id, start_index, end_index)

    # Plot on subfigs
    plot_room_temp(temp_plot, dates, room_temp, user_setpoint, flow_temp, ext_temp)
    plot_hot_water(hot_water_plot, dates, hot_water_temp, hot_water_setpoint, flow_temp)
    plot_tariff(df, tariff_plot, dates, tariff)

    plt.show()

In [ ]:
plot_temp_data(temp_df, house_id, start_index, end_index)